In [1]:
CSV_PATH    = "jobs_final.csv"              # path to your jobs CSV file
ES_URL      = "https://localhost:9200" # Elasticsearch URL
ES_USER     = "elastic"
ES_PASS     = "7S6f9I-bCEFQzsLoQ3IE"
SLEEP_SEC   = 0.0                     # optional delay between rows (0 = max speed)
NUM_WORKERS = 8                       # threads for parallel embedding & indexing
BATCH_SIZE  = 100                     # rows per bulk-index batch


## 1 — Imports

In [2]:
import time
import json
import hashlib

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch

## 2 — Load model

In [3]:
print("Loading JobBERT — this may take a minute on first run...")
jobbert = SentenceTransformer("TechWolf/JobBERT-v2")
DIM = jobbert.get_embedding_dimension()  
EMBEDDING_DIM = DIM * 3                           
print(f"Model ready. Segment dim: {DIM} | Final embedding dim: {EMBEDDING_DIM}")

Loading JobBERT — this may take a minute on first run...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model ready. Segment dim: 1024 | Final embedding dim: 3072


## 3 — Connect to Elasticsearch & create index

In [4]:
es = Elasticsearch(
    ES_URL,
    basic_auth=(ES_USER, ES_PASS),
    verify_certs=False,
    ssl_show_warn=False,
)

In [8]:
INDEX_NAME = "jobs_strat2_normal"

mapping = {
    "mappings": {
        "properties": {
            "job_id":          {"type": "keyword"},
            "normal_job_title":       {"type": "text"},
            "job_description": {"type": "text"},
            "skills":          {"type": "keyword"},
            "embedding": {
                "type":       "dense_vector",
                "dims":       EMBEDDING_DIM,
                "index":      True,
                "similarity": "cosine",
            },
        }
    }
}

if not es.indices.exists(index=INDEX_NAME):
    es.indices.create(index=INDEX_NAME, body=mapping)
    print(f"Created index '{INDEX_NAME}'")
else:
    print(f"Index '{INDEX_NAME}' already exists — skipping creation")

Index 'jobs_strat2_normal' already exists — skipping creation


In [9]:
# Check whether the Elasticsearch index exists and whether it's empty
try:
    if not es.indices.exists(index=INDEX_NAME):
        print(f"Index '{INDEX_NAME}' does not exist.")
    else:
        count = es.count(index=INDEX_NAME)["count"]
        if count == 0:
            print(f"Index '{INDEX_NAME}' is empty (0 documents).")
        else:
            print(f"Index '{INDEX_NAME}' contains {count:,} documents.")
except Exception as e:
    print("Error checking index:", str(e))

Index 'jobs_strat2_normal' contains 50,295 documents.


## 4 — Load & preview CSV

In [10]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows")
df.head(3)

C:\Users\Hagar Hisham Mostafa\AppData\Local\Temp\ipykernel_4120\2083491792.py:1: DtypeWarning: Columns (0: job_link, 1: company, 2: employment_type, 3: job_description, 4: job_location, 5: job_title, 6: post_date, 7: shift, 8: site_name, 9: skills, 10: job_level, 11: job_id, 12: country, 13: city, 14: job_type, 15: Source, 16: Title, 17: Company, 18: Location, 19: Keywords Matched, 20: Skills, 21: Date Posted, 22: URL, 23: Description, 24: years_of_experience, 25: keywords, 26: job_salary, 27: job_function, 28: industry, 29: listing_type, 30: emails, 31: company_link, 32: company_addresses, 33: company_num_employees, 34: company_description, 35: company_logo, 36: company_url, 37: new_skills, 38: job_position, 39: job_skills, 40: job_summary, 41: is_tech, 42: esco_title, 43: esco_uri, 44: esco_matched_skills) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


Loaded 216,336 rows


,job_link,company,employment_type,job_description,job_location,job_title,post_date,shift,site_name,skills,...,Unnamed: 0,job_position,job_skills,job_summary,is_tech,esco_title,esco_uri,esco_confidence,esco_title_score,esco_matched_skills
0,https://www.dice.com/jobs/detail/AUTOMATION-TE...,"Digital Intelligence Systems, LLC","C2H Corp-To-Corp, C2H Independent, C2H W2, 3 M...",Looking for Selenium engineers...must have sol...,"Atlanta, GA",AUTOMATION TEST ENGINEER,NaN,Telecommuting not available|Travel not required,NaN,SEE BELOW,...,NaN,NaN,NaN,NaN,NaN,automation controls engineer,http://data.europa.eu/esco/occupation/70014e9b...,83.85,73.076923,"c, r, r, r"
1,https://www.dice.com/jobs/detail/Information-S...,University of Chicago/IT Services,Full Time,The University of Chicago has a rapidly growin...,"Chicago, IL",Information Security Engineer,NaN,Telecommuting not available|Travel not required,NaN,"linux/unix, network monitoring, incident respo...",...,NaN,NaN,NaN,NaN,NaN,security operations manager,http://data.europa.eu/esco/occupation/ac638802...,75.00,75.000000,NaN
2,https://www.dice.com/jobs/detail/Business-Solu...,"Galaxy Systems, Inc.",Full Time,"GalaxE.SolutionsEvery day, our solutions affec...","Schaumburg, IL",Business Solutions Architect,NaN,Telecommuting not available|Travel not required,NaN,"Enterprise Solutions Architecture, business in...",...,NaN,NaN,NaN,NaN,NaN,solutions architect,http://data.europa.eu/esco/occupation/85e60565...,88.51,80.851064,r


In [11]:
df.columns

Index(['job_link', 'company', 'employment_type', 'job_description',
       'job_location', 'job_title', 'post_date', 'shift', 'site_name',
       'skills', 'job_level', 'job_id', 'country', 'city', 'job_type',
       'Source', 'Title', 'Company', 'Location', 'Keywords Matched', 'Skills',
       'Date Posted', 'URL', 'Description', 'years_of_experience', 'keywords',
       'job_salary', 'job_function', 'industry', 'listing_type', 'emails',
       'company_link', 'company_addresses', 'company_num_employees',
       'company_description', 'company_logo', 'company_url', 'new_skills',
       'Unnamed: 0', 'job_position', 'job_skills', 'job_summary', 'is_tech',
       'esco_title', 'esco_uri', 'esco_confidence', 'esco_title_score',
       'esco_matched_skills'],
      dtype='str')

## 5 — Clean & prepare

In [11]:
def make_job_id(row) -> str:
    """Use job_link as ID; fall back to an MD5 hash of title+company+location."""
    link = str(row.get("job_link", "")).strip()
    if link and link != "nan":
        return link
    seed = f"{row.get('job_title','')}|{row.get('job_company','')}|{row.get('job_location','')}" 
    return "job_" + hashlib.md5(seed.encode()).hexdigest()[:12]


def generate_job_embedding(title: str, description: str, skills: list[str]) -> list[float]:
    """
    Concatenate JobBERT encodings of title, description, and skills
    into a single 3072-dim vector: [title(1024) | desc(1024) | skills(1024)]
    """
    title_emb  = jobbert.encode(title       if title       else "unknown")
    desc_emb   = jobbert.encode(description if description else "")
    skills_emb = jobbert.encode(" ".join(skills) if skills else "general")
    return np.concatenate([title_emb, desc_emb, skills_emb]).tolist()


# Apply cleaning
df["_job_id"]  = df.apply(make_job_id, axis=1)
df["_title"]   = df["esco_title"].fillna("Other").str.strip()
df["_skills"]  = df["new_skills"].apply(lambda x: [s.strip() for s in str(x).split("|") if s.strip()])
df["_description"] = df["job_description"]

before = len(df)
df = df[df["_title"] != ""].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with no title. Ready to embed: {len(df):,}")

Dropped 0 rows with no title. Ready to embed: 216,336


## 6 — Dry run (first 3 rows)

In [12]:
for _, row in df.head(3).iterrows():
    emb = generate_job_embedding(row["_title"], row["_description"], row["_skills"])
    print(f"[OK] '{row['_title']}' → {len(emb)} dims | skills: {row['_skills'][:3]}")

[OK] 'automation controls engineer' → 3072 dims | skills: ['sql', 'unix', 'jenkins']
[OK] 'security operations manager' → 3072 dims | skills: ['unix', 'linux', 'python']
[OK] 'solutions architect' → 3072 dims | skills: ['etl', 'data warehousing', 'data analysis']


## 7 — Bulk embed & index
Embeds and indexes every row directly into Elasticsearch. Errors are collected without stopping the run.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from elasticsearch.helpers import bulk
import threading
import traceback
import time

# Thread-safe counters
_lock   = threading.Lock()
indexed = 0
errors  = []

# Find already-indexed IDs via mget (chunked)
all_ids = df["_job_id"].tolist()
print(f"[DEBUG] Total IDs to check: {len(all_ids):,}")
existing_ids = set()
chunk_size = 1000
for i in range(0, len(all_ids), chunk_size):
    chunk = all_ids[i : i + chunk_size]
    if not chunk:
        continue
    resp = es.mget(index=INDEX_NAME, body={"ids": chunk})
    found = sum(1 for doc in resp.get("docs", []) if doc.get("found"))
    print(f"[DEBUG] mget chunk {i}-{i+len(chunk)}: checked {len(chunk):,}, found {found}")
    for doc in resp.get("docs", []):
        if doc.get("found"):
            existing_ids.add(doc["_id"])

# Filter rows to only those not yet indexed
rows_to_index = [row for _, row in df.iterrows() if row["_job_id"] not in existing_ids]

print(f"[DEBUG] Skipping {len(existing_ids):,} already-indexed jobs. To index: {len(rows_to_index):,}")

if not rows_to_index:
    print("Nothing to do.")
else:
    def embed_and_prepare(row):
        """Embed one row and return an ES action dict (or raise on error)."""
        job_id = row["_job_id"]
        try:
            print(f"[DEBUG] Start embed: {job_id} | title='{row.get('_title')}' | skills={row.get('_skills')}")
            start = time.time()
            embedding = generate_job_embedding(
                row["_title"],
                row["_description"],
                row["_skills"],
            )
            elapsed = time.time() - start
            print(f"[DEBUG] Done embed: {job_id} ({len(embedding)} dims) in {elapsed:.2f}s")
            return {
                "_index": INDEX_NAME,
                "_id":    job_id,
                "_source": {
                    "job_id":            job_id,
                    "normal_job_title":  row["_title"],
                    "job_description":   row["_description"],
                    "skills":            row["_skills"],
                    "embedding":         embedding,
                },
            }
        except Exception as e:
            print(f"[ERROR] embed_and_prepare failed for {job_id}: {e}")
            print(traceback.format_exc())
            raise

    def flush_batch(batch):
        """Bulk-index a list of ES action dicts; return (n_ok, failed_list)."""
        if not batch:
            return 0, []
        print(f"[DEBUG] Flushing batch of {len(batch):,} docs. First id: {batch[0]['_id'] if batch else 'N/A'}")
        ok, failed = bulk(es, batch, raise_on_error=False, stats_only=False)
        print(f"[DEBUG] Bulk returned ok={ok} failed={len(failed) if failed else 0}")
        return ok, failed

    pending_batch = []

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = {pool.submit(embed_and_prepare, row): row["_job_id"] for row in rows_to_index}
        print(f"[DEBUG] Submitted {len(futures):,} tasks to ThreadPoolExecutor (workers={NUM_WORKERS})")
        for future in tqdm(as_completed(futures), total=len(futures), desc="Embedding & indexing"):
            job_id = futures[future]
            try:
                action = future.result()
                batch_to_flush = None
                with _lock:
                    pending_batch.append(action)
                    if len(pending_batch) >= BATCH_SIZE:
                        batch_to_flush = pending_batch[:]
                        pending_batch.clear()
                if batch_to_flush:
                    ok, failed = flush_batch(batch_to_flush)
                    with _lock:
                        indexed += ok
                        if failed:
                            errors.extend(failed if isinstance(failed, list) else [failed])
            except Exception as exc:
                with _lock:
                    err_info = {"job_id": job_id, "error": str(exc), "trace": traceback.format_exc()}
                    errors.append(err_info)
                    print(f"[ERROR] Task failed for {job_id}: {exc}")

    # Flush any remaining docs
    if pending_batch:
        print(f"[DEBUG] Flushing final pending batch of {len(pending_batch):,} docs")
        ok, failed = flush_batch(pending_batch)
        indexed += ok
        if failed:
            errors.extend(failed if isinstance(failed, list) else [failed])

    print(f"\nDone.  Indexed: {indexed:,} | Errors: {len(errors)}")
    if errors:
        print(f"[DEBUG] Sample errors (up to 5): {errors[:5]}")


[DEBUG] Total IDs to check: 216,336
[DEBUG] mget chunk 0-1000: checked 1,000, found 1000
[DEBUG] mget chunk 1000-2000: checked 1,000, found 1000
[DEBUG] mget chunk 2000-3000: checked 1,000, found 1000
[DEBUG] mget chunk 3000-4000: checked 1,000, found 1000
[DEBUG] mget chunk 4000-5000: checked 1,000, found 1000
[DEBUG] mget chunk 5000-6000: checked 1,000, found 1000
[DEBUG] mget chunk 6000-7000: checked 1,000, found 1000
[DEBUG] mget chunk 7000-8000: checked 1,000, found 1000
[DEBUG] mget chunk 8000-9000: checked 1,000, found 1000
[DEBUG] mget chunk 9000-10000: checked 1,000, found 1000
[DEBUG] mget chunk 10000-11000: checked 1,000, found 1000
[DEBUG] mget chunk 11000-12000: checked 1,000, found 1000
[DEBUG] mget chunk 12000-13000: checked 1,000, found 1000
[DEBUG] mget chunk 13000-14000: checked 1,000, found 1000
[DEBUG] mget chunk 14000-15000: checked 1,000, found 1000
[DEBUG] mget chunk 15000-16000: checked 1,000, found 1000
[DEBUG] mget chunk 16000-17000: checked 1,000, found 1000


Embedding & indexing:   0%|          | 0/132979 [00:00<?, ?it/s]

[ERROR] embed_and_prepare failed for https://www.linkedin.com/jobs/view/senior-systems-analyst-at-leidos-3754714962: Unsupported input type: float. Expected one of: str, dict, PIL.Image.Image, np.ndarray, torch.Tensor
[ERROR] embed_and_prepare failed for https://www.linkedin.com/jobs/view/sr-business-analyst-w-d-liquidity-and-payments-technology-exp-must-at-hirekeyz-inc-3693988500: Unsupported input type: float. Expected one of: str, dict, PIL.Image.Image, np.ndarray, torch.Tensor
Traceback (most recent call last):
  File "C:\Users\Hagar Hisham Mostafa\AppData\Local\Temp\ipykernel_4120\3571272380.py", line 42, in embed_and_prepare
    embedding = generate_job_embedding(
        row["_title"],
        row["_description"],
        row["_skills"],
    )
  File "C:\Users\Hagar Hisham Mostafa\AppData\Local\Temp\ipykernel_4120\3218676730.py", line 16, in generate_job_embedding
    desc_emb   = jobbert.encode(description if description else "")
  File "c:\Users\Hagar Hisham Mostafa\Documents\

## 8 — Inspect errors

In [10]:
if errors:
    err_df = pd.DataFrame(errors)
    display(err_df)
    err_df.to_csv("embedding_errors.csv", index=False)
    print("Saved to embedding_errors.csv")
else:
    print("No errors — all jobs indexed successfully.")

,job_id,title,error
0,https://www.dice.com/jobs/detail/AUTOMATION-TE...,automation controls engineer,'_summary'
1,https://www.dice.com/jobs/detail/Information-S...,security operations manager,'_summary'
2,https://www.dice.com/jobs/detail/Business-Solu...,solutions architect,'_summary'
3,https://www.dice.com/jobs/detail/Java-Develope...,Other,'_summary'
4,https://www.dice.com/jobs/detail/DevOps-Engine...,cloud DevOps engineer,'_summary'
...,...,...,...
216331,https://www.linkedin.com/jobs/view/test-engine...,hardware engineer,'_summary'
216332,https://uk.linkedin.com/jobs/view/fire-securit...,IT infrastructure engineer,'_summary'
216333,https://www.linkedin.com/jobs/view/senior-soft...,Other,'_summary'
216334,https://www.linkedin.com/jobs/view/network-aut...,automation controls engineer,'_summary'


Saved to embedding_errors.csv


## 9 — Verify: kNN search test

In [16]:
# Build a query embedding from the first row's title + skills
sample = df.iloc[0]

query_vector = np.concatenate([
    jobbert.encode(sample["_title"]),
    np.zeros(DIM),                                                         # zero-pad description slot
    jobbert.encode(" ".join(sample["_skills"]) if sample["_skills"] else "general"),
]).tolist()

response = es.search(
    index=INDEX_NAME,
    body={
        "knn": {
            "field":          "embedding",
            "query_vector":   query_vector,
            "k":              5,
            "num_candidates": 50,
        },
        "_source": ["job_id", "job_title", "skills"],
    },
)

print(f"Top 5 matches for '{sample['_title']}':")
for hit in response["hits"]["hits"]:
    src = hit["_source"]
    print(f"  [{hit['_score']:.4f}] {src['job_title']}")

Top 5 matches for 'Sr. Hardware Simulation Platform Engineer, Chassis Controls & Powertrain':
  [0.9406] Sr. Hardware Simulation Platform Engineer, Chassis Controls & Powertrain
  [0.8505] Sr. Mechanical Engineer, Vehicle & Firmware, Hardware Test Engineering
  [0.8293] Senior Software Engineer, Embedded Systems/Firmware, Platforms Infrastructure Engineering
  [0.8226] Senior System Software Engineer - Automotive AI Applications
  [0.8222] Sr. Flight Control Systems Engineer


## 10 — Index stats

In [18]:

count = es.count(index=INDEX_NAME)["count"]
print(f"Total documents in '{INDEX_NAME}': {count}")

Total documents in 'jobs': 39650
